In [ ]:
"""
AI-Generated Email Evaluation System
Implements 6 evaluation metrics based on research framework
"""

# ============================================================================
# INSTALLATION & SETUP
# ============================================================================

# Install required packages
!pip install openai anthropic google-generativeai sentence-transformers scikit-learn pandas numpy tenacity -q

import os
import json
import time
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple
from dataclasses import dataclass
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import openai
from anthropic import Anthropic
from tenacity import retry, stop_after_attempt, wait_random_exponential

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    """Configuration for API keys and model selection"""

    # ============================================================================
    # OPENROUTER CONFIGURATION
    # ============================================================================
    # Set your OpenRouter API key here
    OPENROUTER_API_KEY = "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"

    # OpenRouter base URL
    OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

    # Choose your model from OpenRouter's catalog
    # Examples:
    # - "openai/gpt-4o"
    # - "anthropic/claude-sonnet-4"
    # - "google/gemini-2.5-flash"
    # - "anthropic/claude-3.5-haiku"
    # - "openai/gpt-4o-mini"
    # - "google/gemma-3-4b-it"
    EVALUATION_MODEL = "x-ai/grok-4-fast"


    # Semantic entropy settings
    N_SEMANTIC_SAMPLES = 5  # Number of outputs to generate for semantic entropy

    # API Pricing (USD per 1M tokens) - OpenRouter pricing
    PRICING = {
    # OpenAI Models
    'openai/gpt-5': {'input': 1.25, 'output': 10.0},
    'openai/gpt-5-chat': {'input': 1.25, 'output': 10.0},
    'openai/gpt-4o': {'input': 2.5, 'output': 10.0},
    'openai/gpt-4-turbo': {'input': 10.0, 'output': 30.0},
    'openai/gpt-4o-mini': {'input': 0.15, 'output': 0.6},

    # Anthropic Models
    'anthropic/claude-sonnet-4': {'input': 3.0, 'output': 15.0},
    'anthropic/claude-3.5-haiku': {'input': 0.8, 'output': 4.0},
    'anthropic/claude-3-haiku': {'input': 0.25, 'output': 1.25},

    # Google Models
    'google/gemini-2.5-flash': {'input': 0.3, 'output': 2.5},
    'google/gemini-2.5-pro': {'input': 1.25, 'output': 10.0},
    'google/gemma-3-4b-it': {'input': 0.017, 'output': 0.068},

    # Qwen Models
    'qwen/qwen-2.5-72b-instruct': {'input': 0.35, 'output': 0.4},
    'qwen/qwen2.5-vl-72b-instruct': {'input': 0.0, 'output': 0.0},
    'qwen/qwen3-coder-30b-a3b-instruct': {'input': 0.06, 'output': 0.25},

    # Meta Models
    'meta-llama/llama-3.3-70b-instruct': {'input': 0.35, 'output': 0.4},
    'meta-llama/llama-3.1-8b-instruct': {'input': 0.016, 'output': 0.03},

    # Mistral Models
    'mistralai/mistral-small-3.2-24b-instruct:free': {'input': 0.0, 'output': 0.0},

    # xAI Models
    'x-ai/grok-4-fast': {'input': 0.20, 'output': 0.50},

    # Amazon Models
    'amazon/nova-micro-1.0': {'input': 0.035, 'output': 0.14},

    # DeepSeek Models
    'deepseek/deepseek-v3-0324': {'input': 0.24, 'output': 0.84},
    'deepseek/deepseek-v3': {'input': 0.3, 'output': 0.85},

    # Llama Models:
    'sao10k/l3-lunaris-8b': {'input': 0.04, 'output': 0.05},
    }

    @classmethod
    def setup(cls):
        """Setup OpenRouter client"""
        # Configure OpenAI client to use OpenRouter
        client = openai.OpenAI(
            api_key=cls.OPENROUTER_API_KEY,
            base_url=cls.OPENROUTER_BASE_URL,
            default_headers={
                "HTTP-Referer": "https://github.com/yourusername/email-eval",  # Optional
                "X-Title": "Email Evaluation System",  # Optional
            }
        )
        return {
            'openai': client,  # This client works for all OpenRouter models
            'openrouter': client
        }

    @classmethod
    def get_model_cost(cls, model: str, input_tokens: int, output_tokens: int) -> float:
        """Calculate cost for a model call"""
        if model not in cls.PRICING:
            # If model not in pricing dict, return 0 (unknown cost)
            print(f"⚠️  Warning: No pricing info for model '{model}'. Cost tracking disabled.")
            return 0.0

        pricing = cls.PRICING[model]
        input_cost = (input_tokens / 1_000_000) * pricing['input']
        output_cost = (output_tokens / 1_000_000) * pricing['output']
        return input_cost + output_cost

# ============================================================================
# DATA STRUCTURES
# ============================================================================

@dataclass
class EvaluationResult:
    """Store evaluation results for a single email"""
    hallucination_score: int  # Binary: 0 or 1
    cta_quality: int  # 1-5
    language_quality: int  # 1-5
    personalization: int  # 1-5
    human_likeness: int  # 1-5
    instruction_adherence: int  # 1-5

    overall_score: float
    is_acceptable: bool
    detailed_feedback: Dict

    # Cost tracking
    total_cost: float
    input_tokens: int
    output_tokens: int
    api_calls: int

    def to_dict(self):
        return {
            'hallucination_score': self.hallucination_score,
            'cta_quality': self.cta_quality,
            'language_quality': self.language_quality,
            'personalization': self.personalization,
            'human_likeness': self.human_likeness,
            'instruction_adherence': self.instruction_adherence,
            'overall_score': self.overall_score,
            'is_acceptable': self.is_acceptable,
            'total_cost_usd': self.total_cost,
            'input_tokens': self.input_tokens,
            'output_tokens': self.output_tokens,
            'api_calls': self.api_calls,
            'detailed_feedback': self.detailed_feedback
        }

# ============================================================================
# LLM-AS-JUDGE EVALUATOR
# ============================================================================

class LLMEvaluator:
    """
    Uses LLM-as-judge approach for evaluating email quality
    Works with OpenRouter API
    """

    def __init__(self, model: str = Config.EVALUATION_MODEL):
        self.model = model
        self.clients = Config.setup()
        self.client = self.clients['openrouter']  # Use OpenRouter client for all models
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_api_calls = 0
        self.total_cost = 0.0
        self.api_latencies = []

    @retry(wait=wait_random_exponential(multiplier=1, max=60), stop=stop_after_attempt(3))
    def _call_llm(self, prompt: str, score_range: Tuple[int, int] = (1, 5)) -> Tuple[int, int, int]:
        """
        Call LLM via OpenRouter with retry mechanism
        Returns: (score, input_tokens, output_tokens)
        """

        try:
            t_api_start = time.perf_counter()
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Respond ONLY with a JSON object "
                            f"of the form {{\"score\": <integer between {score_range[0]} and {score_range[1]}>}}. "
                            "Do not add any other text or formatting."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0,
                max_tokens=10,
                response_format={"type": "json_object"},
            )
            t_api_end = time.perf_counter()
            self.api_latencies.append(t_api_end - t_api_start)

            input_tokens = response.usage.prompt_tokens
            output_tokens = response.usage.completion_tokens

            # 获取内容
            content = (response.choices[0].message.content or "").strip()
            if not content:
                print("⚠️ Empty response from model, returning -1")
                return -1, input_tokens, output_tokens

            # 尝试解析
            try:
                data = json.loads(content)
            except json.JSONDecodeError:
                print(f"⚠️ JSON decode failed, raw content: {content}")
                return -1, input_tokens, output_tokens

            score = int(data.get("score", -1))

            # 统计 cost
            self.total_input_tokens += input_tokens
            self.total_output_tokens += output_tokens
            self.total_api_calls += 1
            self.total_cost += Config.get_model_cost(self.model, input_tokens, output_tokens)

            return score, input_tokens, output_tokens

        except Exception as e:
            print(f"❌ LLM call failed: {e}")
            raise


    def reset_usage(self):
        """Reset usage counters"""
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_api_calls = 0
        self.total_cost = 0.0

    def get_usage_stats(self) -> Dict:
        """Get current usage statistics"""
        return {
            'total_input_tokens': self.total_input_tokens,
            'total_output_tokens': self.total_output_tokens,
            'total_api_calls': self.total_api_calls,
            'total_cost_usd': round(self.total_cost, 4)
        }

    @staticmethod
    def safe_json_loads(text: str):
        """Robustly parse possibly wrapped JSON from model output."""
        import json, re

        if not text or not text.strip():
            raise ValueError("Empty LLM response")

        s = text.strip()

        # 去掉 ```json ... ``` 包裹
        if s.startswith("```"):
            s = s.strip("` \n")
            if s.lower().startswith("json"):
                s = s[4:].strip()

        # 尝试直接解析
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            pass

        # 提取第一个 { 到最后一个 } 再试
        start, end = s.find("{"), s.rfind("}")
        if 0 <= start < end:
            snippet = s[start:end+1]
            try:
                return json.loads(snippet)
            except json.JSONDecodeError:
                pass

        # 替换单引号为双引号重试
        s2 = re.sub(r"(?<!\\)'", '"', s)
        return json.loads(s2)

    def evaluate_hallucination(self, email: str, user_prompt: str) -> int:
        """Evaluate Hallucination (Binary Score: 0 or 1)"""
        prompt = f"""Evaluate the hallucination of this email on a binary scale of 0 or 1.

        Email:
        {email}

        {f"user_prompt: {user_prompt}" if user_prompt else ""}

        Description: A binary check for whether the model either fabricated information.

        Scoring:
        0: No Critical Failure. The output contains no fabricated information.
        1: Critical Failure Present. The output contains a clear and misleading falsehood that breaks trust. or has made up information not in the JSON.

        Respond only with a valid JSON object:
        {{
        "score": <integer 0 or 1>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(0, 1))
        return score

    def evaluate_cta_quality(self, email: str) -> int:
        """Evaluate Call-to-Action quality (1-5 scale)"""
        prompt = f"""Evaluate the Call-to-Action (CTA) quality of this email on a scale of 1-5.

        Email:
        {email}

        Evaluation Criteria:
        - Clarity: Is the desired action clear?
        - Relevance: Does it align with the email's purpose?
        - Effectiveness: Is it compelling and low-friction?
        - Appropriateness: Does it fit the context?

        Scoring:
        5: Excellent. The CTA is compelling, low-friction, contextually perfect, and directly supports the email's goal.
        4: Good. The CTA is clear and relevant but could be more compelling or better phrased.
        3: Average. The CTA is functional but generic or weak (e.g., "Let me know your thoughts").
        2: Poor. The CTA is vague, confusing, or mismatched with the email's tone/goal.
        1: Very Poor. The CTA is missing, inappropriate, or makes no sense.

        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score

    def evaluate_language_quality(self, email: str) -> int:
        """Evaluate language quality and coherence (1-5 scale)"""
        prompt = f"""Evaluate the language quality and structural coherence of this email on a scale of 1-5.

        Email:
        {email}

        Evaluation Criteria:
        - Grammar and spelling
        - Vocabulary appropriateness
        - Sentence structure variety
        - Conciseness (no unnecessary repetition)
        - Punctuation and formatting

        Scoring:
        5: Excellent - Free of errors with appropriate vocabulary, concise presentation, varied sentences, and proper formatting.
        4: Good - Mostly error-free with minor issues in vocabulary, conciseness, sentence variety, or formatting.
        3: Average - Some major errors present with noticeable issues in vocabulary, repetition, sentence variety, or formatting.
        2: Poor - Numerous errors with inappropriate vocabulary, significant repetition, limited sentence variety, and formatting issues.
        1: Very Poor - Pervasive errors that impede comprehension with inappropriate vocabulary, excessive repetition, and poor formatting.

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score


    def evaluate_personalization(self, email: str, recipient_data: Dict) -> int:
        """Evaluate personalization and personalization (1-5 scale)"""
        prompt = f"""Evaluate how well this email is tailored to the specific recipient and addresses their relevant needs.

        Email:
        {email}

        Recipient Data:
        {json.dumps(recipient_data, indent=2)}

        Evaluation Criteria (50% each):

        1. Recipient-Specific Tailoring
          - Uses at least 2 relevant data points (role, company, activities, signals).
          - Integrates details coherently (not a list or forced insertion).
          - Avoids prohibited info: founding year, employee count, work history >4 years.

        2. Relevance & Value
          - Addresses a real challenge or opportunity specific to the recipient’s role.
          - Provides tangible value or actionable next step for the recipient.
          - Tone and timing fit their current context.

        Scoring guidelines:
        - 5: Highly personalized; clear unique value; deep understanding of recipient.
        - 4: Good personalization; relevant and useful.
        - 3: Basic personalization; somewhat generic but relevant.
        - 2: Minimal personalization; loosely relevant.
        - 1: Generic; no relevance or value.

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score


    def evaluate_human_likeness(self, email: str) -> int:
        """Evaluate how human-like the email sounds (1-5 scale)"""
        prompt = f"""Evaluate whether this email reads like it was written by a real person with natural thought patterns and authentic voice.

        Email:
        {email}


        valuation Criteria:

        - Naturalness: Conversational tone? Appropriate informality? Authentic voice?
        - Lexical Diversity: Balanced vocabulary richness? Natural word variety without forced sophistication?
        - Conciseness: Direct and efficient? Avoids unnecessary verbosity or over-explanation?
        - Emotional Authenticity: Genuine emotional expression? Appropriate sentiment depth and variety?
        - Structural Naturalness: Varied sentence structure? Natural flow without excessive complexity or formulaic patterns?

        Scoring:

        5: Highly Human-like - Conversational and authentic tone; balanced vocabulary with natural uniqueness; concise and direct; genuine emotional depth with varied sentiment; natural sentence variety and flow
        4: Mostly Human-like - Generally natural tone; good vocabulary balance; reasonably concise; authentic emotions present; varied structure with minor formulaic elements
        3: Ambiguous - Somewhat formal or generic tone; unbalanced vocabulary (too repetitive or artificially diverse); moderately verbose; limited emotional range; some overly complex or uniform sentences
        2: Likely AI-generated - Overly formal or polished tone; excessive use of sophisticated vocabulary or high repeatability; verbose with unnecessary elaboration; shallow or mismatched emotional expressions; complex sentence structures or excessive uniformity
        1: Clearly AI-generated - Robotic or excessively formal tone; unnatural vocabulary patterns (overuse of rare words or excessive repetition); extremely verbose with redundant content; generic or contextually inappropriate emotions; highly formulaic or unnaturally complex structures; lacks personal voice entirely

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score

    def evaluate_instruction_adherence(self, email: str, instructions: str, user_prompt: str) -> int:
        """Evaluate adherence to given instructions (1-5 scale)"""
        prompt = f"""Evaluate how well this email adheres to the given instructions.

        Email:
        {email}

        Instructions:
        {instructions}

        User Prompt:
        {user_prompt}

        Evaluation Criteria:
        - Completeness: Includes all required components
        - Accuracy: Follows instructions without deviation
        - Tone & Style: Maintains required tone
        - Alignment with Goals: Achieves intended purpose
        - Avoidance of Unnecessary Content: No irrelevant information

        Scoring:
        5: Excellent - Fully adheres with no deviations
        4: Good - Mostly adheres with minor omissions
        3: Average - Partially adheres, several deviations
        2: Poor - Frequently fails to follow key instructions
        1: Very Poor - Does not follow instructions at all

        Respond only with a valid JSON object:
        {{
        "score": <integer 1-5>
        }}
        """
        score, _, _ = self._call_llm(prompt, score_range=(1, 5))
        return score


# ============================================================================
# MAIN EVALUATION PIPELINE
# ============================================================================

class EmailEvaluationPipeline:
    """
    Complete evaluation pipeline for AI-generated emails
    Works with OpenRouter API
    """

    def __init__(self, model: str = Config.EVALUATION_MODEL):
        # self.entropy_calc = SemanticEntropyCalculator()
        self.llm_eval = LLMEvaluator(model)
        self.clients = Config.setup()
        self.openrouter_client = self.clients['openrouter']

    # -------------------------------------------------------------
    # LAYER 1 — PRE-SCREENING
    # -------------------------------------------------------------
    def prescreen_email(self, email_id: int, email_text: str) -> Tuple[bool, str, int]:
        """
        Layer 1 prescreen:
        - evaluate_hallucination -> binary (0/1)
        - evaluate_language_quality -> 1-5
        Returns:
            (passed: bool, reason: str_or_None, lang_score: int_or_None)
        """
        try:
            # Hallucination (binary)
            hi = self.llm_eval.evaluate_hallucination(email_text)
        except Exception as e:
            print(f"❌ Error during hallucination check for {email_id}: {e}")
            # Conservative approach: treat error as reject with reason
            return False, "hallucination_check_error", None

        if hi == 1:
            return False, "hallucination_detected", None

        try:
            # Language quality
            lq = self.llm_eval.evaluate_language_quality(email_text)
        except Exception as e:
            print(f"❌ Error during language quality check for {email_id}: {e}")
            return False, "language_check_error", None

        if lq < 3:
            return False, f"low_language_quality_{lq}", lq

        # passed
        return True, None, lq

    # -------------------------------------------------------------
    # LAYER 2 — FULL EVALUATION
    # (Hallucination + language removed)
    # -------------------------------------------------------------
    def evaluate_email(
        self,
        email: str,
        instructions: str = "",
        user_prompt: str = "",
        recipient_data: Dict = None,
        context: str = "",
        lang_score: int = None
    ) -> EvaluationResult:
        """
        Comprehensive email evaluation

        Args:
            email: The email text to evaluate
            instructions: Original instructions for email generation
            user_prompt: User instructions for email generation
            recipient_data: Dict with recipient information (role, company, etc.)
            context: Additional context for relevance evaluation
            check_hallucination: Whether to run semantic entropy check (slower)
        """

        start_time = time.time()
        print("Starting evaluation...")
        detailed_feedback = {}

        # ⬅️ 加在这里
        email_api_latencies = []    # ⬅️ 保存 6 次单独 LLM 调用时长
        t_email_start = time.perf_counter()   # ⬅️ 记录 email 评估起始时间

        # Reset usage counters for this evaluation
        self.llm_eval.reset_usage()

        # 1. CTA Quality
        print("Evaluating CTA quality...")
        t1 = time.perf_counter()
        cta_score = self.llm_eval.evaluate_cta_quality(email)
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 2))
        print(f"CTA score: {cta_score}")

        # 2. Personalization
        print("Evaluating personalization...")
        t1 = time.perf_counter()
        if recipient_data:
            per_score = self.llm_eval.evaluate_personalization(email, recipient_data)
        else:
            per_score = -1  # Default when no recipient data
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 2))
        print(f"Personalization score: {per_score}")

        # 3. Human-likeness
        print("Evaluating human-likeness...")
        t1 = time.perf_counter()
        hl_score = self.llm_eval.evaluate_human_likeness(email)
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 2))
        print(f"Human-likeness score: {hl_score}")

        # 4. Instruction Adherence
        print("Evaluating instruction adherence...")
        t1 = time.perf_counter()
        if instructions:
            ia_score = self.llm_eval.evaluate_instruction_adherence(email, instructions, user_prompt)
        else:
            ia_score = -1  # Default when no instruction
        t2 = time.perf_counter()
        email_api_latencies.append(round(t2 - t1, 2))
        print(f"Instruction Adherence score: {ia_score}")

        # Get usage statistics
        usage_stats = self.llm_eval.get_usage_stats()

        # Use provided language score
        if lang_score is None:
            # Defensive fallback (should not happen if pipeline used correctly)
            print("⚠️ Warning: lang_score not provided to evaluate_email(); calling evaluator as fallback.")
            t0 = time.perf_counter()
            lang_score = self.llm_eval.evaluate_language_quality(email)
            t1 = time.perf_counter()
            api_latencies.append(round(t1 - t0, 3))
        else:
            # Record zero latency placeholder for language since it's precomputed upstream
            api_latencies.insert(0, 0.0)

        # Compute overall: mean of 5 metrics (exclude hallucination)
        scores = [cta_score, lang_score, per_score, hl_score, ia_score]
        overall_score = float(np.mean(scores))

        # Check acceptability conditions
        # Unacceptable if: HI == 1 OR AVG(scores) <= 3 OR LQ <= 2
        # Also consider hallucination_score = -1 as potentially unacceptable
        is_acceptable = (
            (hi_score == 0 or hi_score == -1) and # Treat error as potentially acceptable if other scores are good
            overall_score >= 3 and
            lq_score >= 2
        )

        elapsed_time = time.time() - start_time  # ⬅️ Added: calculate elapsed time
        detailed_feedback['runtime_seconds'] = elapsed_time  # ⬅️ Added: store runtime in feedback
        t_email_end = time.perf_counter()
        detailed_feedback["email_total_time"] = t_email_end - t_email_start    # ⬅️ 保存 email 总时间
        detailed_feedback["latencies"] = email_api_latencies                  # ⬅️ 保存 6 次 latency

        print(f"\nEvaluation complete!")
        print(f"💰 Cost: ${usage_stats['total_cost_usd']:.4f}")
        print(f"📊 Tokens: {usage_stats['total_input_tokens']} in / {usage_stats['total_output_tokens']} out")
        print(f"🔄 API Calls: {usage_stats['total_api_calls']}")
        print(f"⏱️ Runtime: {elapsed_time:.2f} seconds")  # ⬅️ Added: print runtime

        return EvaluationResult(
            hallucination_score=hi_score,
            cta_quality=cta_score,
            language_quality=lq_score,
            personalization=per_score,
            human_likeness=hl_score,
            instruction_adherence=ia_score,
            overall_score=overall_score,
            is_acceptable=is_acceptable,
            detailed_feedback=detailed_feedback,
            total_cost=usage_stats['total_cost_usd'],
            input_tokens=usage_stats['total_input_tokens'],
            output_tokens=usage_stats['total_output_tokens'],
            api_calls=usage_stats['total_api_calls']
        )

    # -------------------------------------------------------------
    # LAYER 3 — APPLY TO BATCH
    # -------------------------------------------------------------
    def evaluate_batch(self,
                       emails: List[Dict],
                       save_rejected_path: str = "rejected_ids.json") -> pd.DataFrame:
        """
        Evaluate a list of emails.
        Each email dict should contain:
          - 'id' (optional)
          - 'email' (string)
          - 'instructions' (optional)
          - 'user_prompt' (optional)
          - 'recipient_data' (optional)
          - 'context' (optional)
        """

        pipeline_results = []
        rejected_records = []
        total_cost = 0.0
        batch_start = time.time()

        for idx, item in enumerate(emails):
            email_id = item.get("id", idx)
            email_text = item.get("email", "")
            instructions = item.get("instructions", "")
            user_prompt = item.get("user_prompt", "")
            recipient_data = item.get("recipient_data", {})
            context = item.get("context", "")

            print("\n" + "="*60)
            print(f"Processing email {idx+1}/{len(emails)} — id: {email_id}")
            print("="*60)

            # -------------------------
            # Layer 1: Pre-screening
            # -------------------------
            try:
                passed, reason, lang_score = self.prescreen_email(email_id, email_text)
            except Exception as e:
                print(f"❌ Prescreen error for {email_id}: {e}")
                passed = False
                reason = "prescreen_exception"
                lang_score = None

            if not passed:
                # record rejection and skip full evaluation
                print(f"❌ Rejected {email_id} during Layer 1: {reason}")
                rejected_records.append({"id": email_id, "reason": reason})
                continue

            # Passed prescreen; lang_score contains the language quality (1-5)
            print(f"✅ Passed prescreen. Language score: {lang_score}")

            # -------------------------
            # Layer 2: Full evaluation (reuse lang_score)
            # -------------------------
            try:
                result: EvaluationResult = self.evaluate_email(
                    email=email_text,
                    instructions=instructions,
                    user_prompt=user_prompt,
                    recipient_data=recipient_data,
                    context=context,
                    lang_score=lang_score
                )
            except Exception as e:
                print(f"❌ Error during full evaluation for {email_id}: {e}")
                # If full evaluation fails, record as rejected to be safe
                rejected_records.append({"id": email_id, "reason": "evaluation_failure"})
                continue

            pipeline_results.append({
                "email_id": email_id,
                **result.to_dict(),
                "precomputed_language_score": lang_score
            })

            total_cost += result.total_cost

        # Save rejected records to JSON
        try:
            with open(save_rejected_path, "w", encoding="utf-8") as f:
                json.dump(rejected_records, f, indent=2)
            print(f"\nSaved {len(rejected_records)} rejected emails → {save_rejected_path}")
        except Exception as e:
            print(f"❌ Failed to save rejected JSON: {e}")

        batch_elapsed = time.time() - batch_start
        print(f"\nBatch completed in {batch_elapsed:.2f}s — Evaluated {len(pipeline_results)} emails, Rejected {len(rejected_records)}")
        print(f"💰 Total estimated cost across evaluated emails: ${total_cost:.4f}")

        df = pd.DataFrame(pipeline_results)
        return df



The following section is the same as above only with shorter prompt.

DO NOT RUN THE ABOVE AND THIS ONE AT THE SAME TIME.

In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

def load_test_data_from_file(file_path: str) -> List[Dict]:
    """
    Load test data from an external file.

    Supported formats:
    - JSON (.json): a JSON array containing test data

    Args:
        file_path: Path to the data file

    Returns:
        List of test data dictionaries
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ File not found: {file_path}")

    file_ext = os.path.splitext(file_path)[1].lower()

    try:
        if file_ext == '.json':
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
        else:
            raise ValueError(f"❌ Unsupported file format: {file_ext}. Supported format: .json")

        # Validate data structure
        required_fields = ['email']
        for i, item in enumerate(data):
            for field in required_fields:
                if field not in item:
                    raise ValueError(f"❌ Missing required field '{field}' in item {i}")

        print(f"✅ Successfully loaded {len(data)} test samples")
        return data

    except Exception as e:
        print(f"❌ Failed to load test data: {e}")
        raise


# ============================================================================
# MAIN TEST FUNCTIONS
# ============================================================================

def test_spreadsheet_examples(file_path: str):
    """Run evaluation pipeline on external test data from a JSON file"""

    print("\n" + "="*80)
    print("🧪 TESTING WITH EXTERNAL TEST DATA")
    print("="*80)

    # Load test data
    print(f"📁 Loading test data from file: {file_path}")
    test_data = load_test_data_from_file(file_path)

    if not test_data:
        print("❌ No test data found")
        return None

    # Initialize the evaluation pipeline
    try:
        pipeline = EmailEvaluationPipeline(model=Config.EVALUATION_MODEL)
    except Exception as e:
        print(f"\n❌ Error: Unable to initialize evaluation pipeline - {e}")
        return None

    # Evaluate all examples
    results_list = []

    for example in test_data:
        print(f"\n{'='*80}")
        print(f"📧 EVALUATING EXAMPLE {example.get('id', 'Unknown')}")
        recipient_data = example.get('recipient_data', {})
        print(f"Recipient: {recipient_data.get('name', 'Unknown')} - {recipient_data.get('role', 'Unknown')}")
        print(f"{'='*80}")

        # Print the email being evaluated
        print("\n📧 EMAIL BEING EVALUATED:")
        print("-" * 80)
        print(example['email'])
        print("-" * 80)

        # Print the evaluation context
        # print("\n📋 EVALUATION CONTEXT:")
        # print(f"Recipient Data: {recipient_data}")
        # print(f"Context: {example.get('context', 'No context')}")
        # print(f"Instructions: {example.get('instructions', 'No instructions provided')}")
        # print(f"User Prompt: {example.get('user_prompt', 'No user prompt provided')}")


        result = pipeline.evaluate_email(
            email=example['email'],
            instructions=example.get('instructions', ''),
            user_prompt=example.get('user_prompt', ''),
            recipient_data=recipient_data,
            context=example.get('context', ''),
            check_hallucination=False  # disable for faster testing
        )

        results_list.append({
            'example_id': example.get('id', 'Unknown'),
            # 'recipient': recipient_data.get('name', 'Unknown'), # Removed recipient
            'overall_score': result.overall_score,
            'acceptable': result.is_acceptable,
            'hallucination': result.hallucination_score,
            'cta_quality': result.cta_quality,
            'language_quality': result.language_quality,
            'personalization': result.personalization,
            'human_likeness': result.human_likeness,
            'instruction_adherence': result.instruction_adherence,
            'cost_usd': result.total_cost,
            'latencies': result.detailed_feedback.get("latencies"),
            'email_total_time': result.detailed_feedback.get("email_total_time")
        })

    # Summary
    print("\n\n" + "="*80)
    print("📊 TEST RESULTS SUMMARY")
    print("="*80)

    df = pd.DataFrame(results_list)
    print(df.to_string(index=False))

    total_cost = df['cost_usd'].sum()
    avg_score = df['overall_score'].mean()
    acceptable_count = df['acceptable'].sum()

    print(f"\n{'='*80}")
    print(f"💰 Total Cost: ${total_cost:.4f}")
    print(f"💰 Average Cost per Email: ${total_cost/len(test_data):.4f}")
    print(f"💰 Estimated Cost for 10,000 emails: ${(total_cost/len(test_data))*10000:.2f}")
    # === Global latency stats ===
    lat_arr = np.array(pipeline.llm_eval.api_latencies)

    print("\n⏱️ API Latency Distribution (ALL CALLS)")
    print(f"P50: {np.percentile(lat_arr, 50):.4f} s")
    print(f"P95: {np.percentile(lat_arr, 95):.4f} s")
    print(f"P99: {np.percentile(lat_arr, 99):.4f} s")

    # Total batch time
    print(f"\n🕒 Total Batch Runtime: {df['email_total_time'].sum():.2f} s")

    print(f"\n📈 Average Overall Score: {avg_score:.2f}/10")
    print(f"✅ Acceptable Emails: {acceptable_count}/{len(test_data)} ({acceptable_count/len(test_data)*100:.0f}%)")
    print(f"{'='*80}\n")

    return df


def analyze_specific_example(file_path: str, example_id: int = 0):
    """Run detailed evaluation on a specific example"""

    print("\n" + "="*80)
    print(f"🔍 DETAILED ANALYSIS: EXAMPLE {example_id}")
    print("="*80)

    # Load test data
    test_data = load_test_data_from_file(file_path)

    # Find the specific example
    example = None
    for item in test_data:
        if item.get('id') == example_id:
            example = item
            break

    if not example:
        print(f"❌ Example with ID {example_id} not found")
        return None

    print("\n📧 EMAIL BEING EVALUATED:")
    print("-" * 80)
    print(example['email'])
    print("-" * 80)

    print("\n📋 EVALUATION CONTEXT:")
    recipient_data = example.get('recipient_data', {})
    print(f"Recipient: {recipient_data.get('name', 'Unknown')}")
    print(f"Role: {recipient_data.get('role', 'Unknown')}")
    print(f"Company: {recipient_data.get('company', 'Unknown')}")
    print(f"Context: {example.get('context', 'No context')}")

    print("\n📝 INSTRUCTIONS GIVEN:")
    print(example.get('instructions', 'No instructions provided'))

    pipeline = EmailEvaluationPipeline(model=Config.EVALUATION_MODEL)

    result = pipeline.evaluate_email(
        email=example['email'],
        instructions=example.get('instructions', ''),
        user_prompt=example.get('user_prompt', ''),
        recipient_data=recipient_data,
        context=example.get('context', ''),
        check_hallucination=False
    )

    print("\n" + "="*80)
    print("📊 EVALUATION RESULTS")
    print("="*80)
    print(f"Overall Score: {result.overall_score:.1f}/10")
    print(f"Acceptable: {'✅ YES' if result.is_acceptable else '❌ NO'}")
    print(f"\nBreakdown:")
    print(f"  CTA Quality: {result.cta_quality}/10")
    print(f"  Language Quality: {result.language_quality}/10")
    print(f"  personalization: {result.personalization}/10")
    print(f"  Human-likeness: {result.human_likeness}/10")
    print(f"  Instruction Adherence: {result.instruction_adherence}/10")
    print(f"\n💰 Cost: ${result.total_cost:.4f}")

    print("\n" + "="*80)
    print("💬 DETAILED FEEDBACK")
    print("="*80)
    for criterion, feedback in result.detailed_feedback.items():
        print(f"\n{criterion.upper()}:")
        if isinstance(feedback, dict):
            for key, value in feedback.items():
                print(f"  {key}: {value}")
        else:
            print(f"  {feedback}")

    return result


# ============================================================================
# MAIN EXECUTION ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    print("\n🚀 Starting Email Evaluation System...")

    # Hardcoded file path for Colab
    file_path = "/content/2 email.json"

    # Choose the mode you want: batch or specific analysis
    # Set this manually for now
    RUN_BATCH_TEST = True   # Set to False if you want to analyze a specific example
    EXAMPLE_ID = 0          # Only used if RUN_BATCH_TEST is False

    if RUN_BATCH_TEST:
        print(f"\n🧪 Running batch test from: {file_path}")
        results_df = test_spreadsheet_examples(file_path=file_path)
    else:
        print(f"\n🔍 Running detailed analysis on example ID {EXAMPLE_ID}")
        analyze_specific_example(file_path=file_path, example_id=EXAMPLE_ID)